# Xem câu hỏi và câu trả lời của các mô hình

Đọc manifest (segment), câu trả lời của mô hình (`out/*.jsonl` do `tools/ask_models.py` ghi), nhãn người (nếu có) và bộ câu hỏi `data/config.json`.
Sửa các đường dẫn ở cell dưới rồi chạy hết.

In [ ]:
import glob, json, os, sys, collections
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "tools"))
from ask_models import load_jsonl, questions
import conf
C = conf.load([])          # thư mục lấy từ paths trong config/ (vd. conf.load(["paths.answers=out2"]))
from audit_questions import majority

MANIFESTS = os.path.join(conf.path(C, "manifests"), "*.jsonl")
ANSWERS   = sorted(glob.glob(os.path.join(conf.path(C, "answers"), "*.jsonl")))   # mỗi file một mô hình
HUMANS    = os.path.join(ROOT, "humans.jsonl")                         # bỏ qua nếu chưa có
TAG       = "turntaking"

# Bộ lọc cho phần xem từng segment (dùng khi không có ipywidgets)
FILTER_EVENT    = None      # vd. "MOT"
FILTER_BUCKET   = None      # vd. "interview"
ONLY_DISAGREE   = True      # chỉ các mẫu mô hình bất đồng
FOCUS_QUESTION  = "tcontent.event"
MAX_SHOW        = 10

In [ ]:
import pandas as pd
segs = {r["event_id"]: r for p in sorted(glob.glob(MANIFESTS)) for r in load_jsonl(p)}
QS = questions(TAG)
QTITLE = {k: q["title"] for k, q in QS}
QLABEL = {k: dict(zip(q["codes"], [o[0] for o in q["opts"]])) for k, q in QS}
models = {os.path.splitext(os.path.basename(p))[0]: {r["sample_id"]: r for r in load_jsonl(p)} for p in ANSWERS}
hum = collections.defaultdict(lambda: collections.defaultdict(list))
if os.path.exists(HUMANS):
    for r in load_jsonl(HUMANS):
        for k, v in r["answers"].items():
            hum[r["sample_id"]][k].append(v)
human = {sid: {k: majority([x for x in v if x != "UNK"]) or (v[0] if len(set(v)) == 1 else None) for k, v in d.items()} for sid, d in hum.items()}
SRC = dict(models, **({"người": {sid: {"answers": a} for sid, a in human.items()}} if human else {}))

rows = []
for name, m in SRC.items():
    for sid, r in m.items():
        for k, v in r["answers"].items():
            rows.append({"source": name, "sample_id": sid, "q": k, "a": v})
A = pd.DataFrame(rows, columns=["source", "sample_id", "q", "a"])
if A.empty:
    raise SystemExit(f"Chưa có câu trả lời nào trong {conf.path(C, 'answers')}: chạy tools/ask_models.py trước.")
print(f"{len(segs)} segment, {len(QS)} câu hỏi, nguồn trả lời: {list(SRC)}")
pd.DataFrame([{"mô hình": n, "số mẫu": len(m), "lỗi parse %": 100 * sum(r.get("parse_error", False) for r in m.values()) / max(len(m), 1),
               "UNK %": 100 * (A[(A.source == n)].a == "UNK").mean()} for n, m in models.items()]).round(1)

## Phân bố đáp án theo từng câu hỏi

In [ ]:
import matplotlib.pyplot as plt
qs = [k for k, _ in QS if k in set(A.q)]
cols = 3
fig, axes = plt.subplots((len(qs) + cols - 1) // cols, cols, figsize=(16, 3.2 * ((len(qs) + cols - 1) // cols)), squeeze=False)
for ax, k in zip(axes.flat, qs):
    t = A[A.q == k].groupby(["a", "source"]).size().unstack(fill_value=0)
    t = (t / t.sum()).reindex([c for c in QLABEL[k] if c in t.index])
    t.plot.bar(ax=ax, width=0.8, legend=ax is axes.flat[0])
    ax.set_title(k, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("tỉ lệ"); ax.tick_params(axis="x", rotation=30, labelsize=8)
for ax in list(axes.flat)[len(qs):]:
    ax.axis("off")
plt.tight_layout()

## Đồng thuận giữa các nguồn theo câu hỏi

Sắp từ thấp đến cao: câu ở đầu bảng là câu máy trả lời không ổn định (hoặc lệch người).

In [ ]:
W = A.pivot_table(index=["sample_id", "q"], columns="source", values="a", aggfunc="first")
names = list(W.columns)
agree = {}
for i, x in enumerate(names):
    for y in names[i + 1:]:
        sub = W[[x, y]].dropna()
        agree[f"{x} ~ {y}"] = (sub[x] == sub[y]).groupby(level="q").mean() * 100
pd.DataFrame(agree).round(0).sort_values(list(agree)[0] if agree else [], ascending=True) if agree else "cần ít nhất 2 nguồn trả lời"

## Ma trận nhầm lẫn cho một câu

So mô hình với người. Nếu chưa có nhãn người và câu là `tcontent.event` thì so với `event_auto` (từ VAD và diarization).

In [ ]:
ref_name = "người" if human else "event_auto"
for name in models:
    m = models[name]
    pairs = []
    for sid, r in m.items():
        ref = human.get(sid, {}).get(FOCUS_QUESTION) if human else (segs.get(sid, {}).get("event_auto") if FOCUS_QUESTION == "tcontent.event" else None)
        if ref and r["answers"].get(FOCUS_QUESTION):
            pairs.append((ref, r["answers"][FOCUS_QUESTION]))
    if pairs:
        cm = pd.crosstab(pd.Series([p[0] for p in pairs], name=ref_name), pd.Series([p[1] for p in pairs], name=name))
        print(f"\n{name}: khớp {100 * sum(a == b for a, b in pairs) / len(pairs):.0f}% trên {len(pairs)} mẫu")
        display(cm)

## Xem từng segment

Audio, transcript, mốc thời gian và bảng trả lời. Ô khác nhau giữa các nguồn được tô vàng.

In [ ]:
from IPython.display import Audio, HTML, display

def disagree(sid):
    return any(len({SRC[n][sid]["answers"].get(k) for n in SRC if sid in SRC[n]} - {None}) > 1 for k, _ in QS)

def show(sid):
    s = segs[sid]
    display(HTML(f"<h4>{sid} · event_auto=<b>{s.get('event_auto')}</b> · user={s.get('user')} · bucket={s.get('bucket')}</h4>"
                 f"<p>Lời người dùng: <b>{s.get('user_text', '')}</b><br>times={s.get('times')} · next_auto={s.get('next_auto')} · "
                 f"gap={s.get('gap_after_sec_auto')} · overlap={s.get('overlap_auto')} · file_labels={s.get('file_labels')}</p>"))
    p = os.path.join(ROOT, s["audio"]) if s.get("audio") else None
    if p and os.path.exists(p):
        display(Audio(p))
    tab = pd.DataFrame({n: {k: SRC[n].get(sid, {}).get("answers", {}).get(k) for k, _ in QS} for n in SRC})
    tab.insert(0, "câu hỏi", [QTITLE[k] for k in tab.index])
    diff = tab[list(SRC)].nunique(axis=1, dropna=True) > 1
    display(tab.style.apply(lambda r: ["background:#fff3b0" if diff[r.name] and c in SRC else "" for c in r.index], axis=1))
    for n in models:
        why = models[n].get(sid, {}).get("reasoning")
        if why:
            display(HTML(f"<details><summary>Suy nghĩ của {n}</summary><pre style='white-space:pre-wrap'>{why}</pre></details>"))

def pick(event=FILTER_EVENT, bucket=FILTER_BUCKET, only_disagree=ONLY_DISAGREE):
    ids = [sid for sid in segs if any(sid in m for m in SRC.values())]
    ids = [i for i in ids if event is None or segs[i].get("event_auto") == event]
    ids = [i for i in ids if bucket is None or bucket in (segs[i].get("bucket") or [])]
    return [i for i in ids if not only_disagree or disagree(i)]

ids = pick()
print(f"{len(ids)} segment khớp bộ lọc")
for sid in ids[:MAX_SHOW]:
    show(sid)

### Bộ lọc tương tác (cần `ipywidgets`)

In [ ]:
try:
    import ipywidgets as w
    evs = [None] + sorted({s.get("event_auto") for s in segs.values()})
    bks = [None] + sorted({b for s in segs.values() for b in (s.get("bucket") or [])})
    ev_w, bk_w, dis_w = w.Dropdown(options=evs, description="event"), w.Dropdown(options=bks, description="bucket"), w.Checkbox(value=ONLY_DISAGREE, description="chỉ bất đồng")
    sel = w.Dropdown(description="segment")
    out = w.Output()
    def refresh(*_):
        sel.options = pick(ev_w.value, bk_w.value, dis_w.value) or [None]
    def render(*_):
        out.clear_output()
        with out:
            if sel.value: show(sel.value)
    for x in (ev_w, bk_w, dis_w): x.observe(refresh, "value")
    sel.observe(render, "value")
    refresh(); display(w.HBox([ev_w, bk_w, dis_w]), sel, out)
except ImportError:
    print("Không có ipywidgets: dùng các biến FILTER_* ở đầu notebook.")